## Initialize Spark and Load Dataset
Initializing Spark and Load the credit card transaction dataset

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/workspace/default/creditcard-fraud/creditcard.csv")

display(df)


### Class Distribution
Count the number of normal and fraudulent transactions to show class imbalance in the dataset.


In [0]:
df.groupBy('Class').count().show()

### Summary Statistics
Display count, mean, standard deviation, min, and max for selected features (V1, V2, V3) to understand their distributions.


In [0]:
df_select = df.select('V1','V2', 'V3').summary().show()

### Summary Statistics for Fraudulent Transactions
Display count, mean, standard deviation, min, and max for selected features (V1, V2, V3) specifically for fraud cases.


In [0]:
df.filter(df.Class == 1).select('V1', 'V2', 'V3').summary().show()

### Feature Correlation with Class
Compute Pearson correlation between each feature (V1–V28) and the target variable to identify predictive features.


In [0]:
for v in ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28']:
  print(v, df.stat.corr(v, 'Class'))

### Selected Important Features
Display the features identified as most predictive of fraud (V3, V7, V10, V12, V14, V17) along with the target variable.


In [0]:
important_features =['V3','V7','V10','V12','V14','V17']
display(df.select(important_features + ['Class']))

### Assemble Features for ML
Combine the selected features and transaction amount into a single feature vector for modeling.


In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=[ 'V3','V7','V10','V12','V14','V17','Amount'], outputCol= 'features')
df_ml = assembler.transform(df)




### Train/Test Split
Split the dataset into training (70%) and testing (30%) sets for model training and evaluation.


In [0]:
train_df, test_df = df_ml.randomSplit([0.7, 0.3], seed = 42)
print('train_df', train_df.count())
print('test_df', test_df.count())

### Train Logistic Regression Model
Train a logistic regression model using the feature vector to predict fraudulent transactions.


In [0]:
from pyspark.ml.classification import LogisticRegression
lr = LogisticRegression(featuresCol = 'features', labelCol= 'Class', maxIter= 10)
lr_model = lr.fit(train_df)



### Model Predictions
Generate predictions on the test dataset and display actual vs predicted labels.


In [0]:
predictions = lr_model.transform(test_df)
display(predictions.select('features', 'Class','prediction'))



### Model Evaluation (AUC)
Evaluate the model’s performance using the Area Under the ROC Curve (AUC), which is suitable for imbalanced datasets.


In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
evaluator = BinaryClassificationEvaluator( labelCol= "Class", rawPredictionCol= 'rawPrediction', metricName= 'areaUnderROC')
auc = evaluator.evaluate(predictions)
print('Auc', auc)

### Feature Coefficients
Display the logistic regression coefficients to understand the impact of each feature on fraud prediction.


In [0]:
import pandas as pd 
coefficients = pd.DataFrame(
    list(zip(['V3','V7','V10','V12','V14','V17', 'Amount'], lr_model.coefficients)),
    columns = ['features', 'Coefficients']
)
display(coefficients)

### Class Distribution Visualization
Bar chart showing the number of normal and fraudulent transactions to highlight class imbalance.


In [0]:
from pyspark.sql.functions import col
import matplotlib.pyplot as plt

counts = df.groupBy('Class').count().toPandas()

plt.bar(counts['Class'].astype(str), counts['count'], color =['red', 'green'])
plt.xlabel("Class (0 = Normal, 1 = Fraud)")
plt.ylabel("Number of Transactions")
plt.title('Class Distribution')
plt.show()

### Correlation Heatmap
Visualize correlations between all numeric features and the target variable (Class) to identify patterns and relationships.


In [0]:
import seaborn as sns
numeric_cols = ['V1','V2','V3','V4','V5','V6','V7','V8','V9','V10','V11','V12','V13','V14','V15','V16','V17','V18','V19','V20','V21','V22','V23','V24','V25','V26','V27','V28','Amount', 'Class']

df_numeric = df.select(numeric_cols).toPandas()

corr = df_numeric.corr()
plt.figure(figsize=(20,10))
sns.heatmap(corr, cmap= 'coolwarm', center= 0)
plt.title('Corrolation Heatmap')
plt.show()

### ROC Curve Visualization
Plot the Receiver Operating Characteristic (ROC) curve to visualize the model's ability to distinguish between fraudulent and normal transactions.


In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

preds = predictions.select("Class","probability").toPandas()

y_true = preds['Class']
y_scores = preds['probability'].apply(lambda x: float(x[1]))

fpr, tpr, thresholds = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='blue', lw=2, label='ROC curve (AUC = %0.3f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='red', lw=1, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()
